# 01 · Báo cáo 1 (Slides 1–7): Tổng quan dữ liệu, ERD, Timeline & Nhãn mua (`TARGETS`)

Đối chiếu trực tiếp với **Báo cáo 1 (`MBD MINI UNDERSTANDING DATA.pdf`, Slides 1–7)**:
- **Slide 2 (ERD & Kiểm kê):** Cấu trúc 5 bảng (`CLIENT_SPLIT`, `TARGETS`, `TRX`, `GEO`, `DIALOG`), khóa nối và kiểm tra join (`src/audit_mbd_inventory.py`, `src/audit_mbd_join_questions.py`, `src/scripts/audit_insights_checklist.py`).
- **Slide 3 (Timeline):** Các nguồn dữ liệu bắt đầu ở các thời điểm khác nhau (`src/audit_mbd_temporal_coverage.py`).
- **Slide 4 (`TARGETS` overview):** Tỷ lệ dương 4 nhãn (`target_1..4`), `y_any ≈ 1,069%`, `10,043%` NaN `diff_trans_date` gắn với `trans_count = 0` (`src/audit_any_product_cohorts.py`, `src/scripts/check_trans_count_target_history.py`).
- **Slide 5 (`y_any` theo tháng):** Diễn biến tỷ lệ mua 12 tháng (`02/2022–01/2023`).
- **Slide 6 (Tỷ lệ mua quay lại):** So sánh khách đã mua ở bất kỳ tháng trước đó với tháng liền trước `T-1`.
- **Slide 7 (Số tháng có mua / khách):** `89,61%` không mua, `10,39%` (`10.411` khách) từng mua, `81,60%` người mua chỉ mua đúng 1 tháng.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "src/scripts"))


def run_script(rel_path: str, *args: str) -> None:
    """Run a repository script with the active Python interpreter."""
    script = ROOT / rel_path
    assert script.exists(), f"Missing script: {script}"
    cmd = [sys.executable, str(script), *args]
    print(f"$ {' '.join(cmd)}")
    subprocess.run(cmd, cwd=ROOT, check=True)


def show_chart(rel_path: str, caption: str | None = None, width: int = 920) -> None:
    """Display a generated chart image inline if present."""
    path = ROOT / rel_path
    if caption:
        display(Markdown(f"**{caption}** (`{rel_path}`)"))
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"_Chưa có file ảnh `{rel_path}` — chạy cell tạo biểu đồ phía trên._"))


def show_table(rel_path: str, nrows: int = 12) -> None:
    """Display a generated CSV table inline if present."""
    path = ROOT / rel_path
    if path.exists():
        display(pd.read_csv(path, nrows=nrows))
    else:
        display(Markdown(f"_Chưa có bảng `{rel_path}`._"))

## 1. Kiểm kê dữ liệu, khóa nối & xem mẫu (Slide 2)

In [ ]:
# Xem nhanh dữ liệu mẫu trong sample_data/ (1.000 khách hàng)
sample_dir = ROOT / "sample_data"
for table in ["clients", "targets", "trx", "geo", "dialog"]:
    files = sorted((sample_dir / table).glob("*.parquet"))
    if files:
        df_s = pd.read_parquet(files[0])
        print(f"{table:8s} | sample shape={df_s.shape} | columns={list(df_s.columns)}")

In [ ]:
# Chạy lại kiểm kê toàn bộ raw & đối soát khóa nối (tùy chọn, cần data/MBD-mini)
RUN_RAW_AUDITS = False
if RUN_RAW_AUDITS:
    run_script("src/audit_mbd_inventory.py")
    run_script("src/audit_mbd_join_questions.py")
    run_script("src/scripts/audit_insights_checklist.py")

show_table("outputs/eda_insights_20261119/join_audit.csv")

## 2. Độ phủ thời gian của các nguồn dữ liệu (Slide 3)

In [ ]:
RUN_TEMPORAL_AUDIT = False
if RUN_TEMPORAL_AUDIT:
    run_script("src/audit_mbd_temporal_coverage.py")

show_table("docs/report/tables/cross_modal/dataset_timeline.csv")
show_chart("docs/report/figures/cross_modal/dataset_timeline.png", "Slide 3 · Các nguồn bắt đầu ở các thời điểm khác nhau")

## 3. Tổng quan 4 nhãn sản phẩm, `y_any` và `diff_trans_date` NaN (Slide 4)

In [ ]:
RUN_TARGET_AUDITS = False
if RUN_TARGET_AUDITS:
    run_script("src/audit_any_product_cohorts.py")
    run_script("src/scripts/check_trans_count_target_history.py")

show_table("docs/report/tables/targets/target_positive_rates.csv")
show_table("docs/report/tables/targets/target_null_patterns.csv")

## 4. Xu hướng `y_any` theo tháng, Tỷ lệ mua quay lại & Số tháng có mua (Slides 5–7)

In [ ]:
REGENERATE_TARGET_CHARTS = False
if REGENERATE_TARGET_CHARTS:
    run_script("outputs/repeat_purchase_review/analyze.py")
    run_script("outputs/trx_charts/plot_y_any.py")
    run_script("outputs/trx_charts/plot_returning_monthly_comparison.py")
    run_script("outputs/trx_charts/plot_purchase_frequency.py")

show_table("outputs/eda_insights_20261119/label_monthly.csv", nrows=12)
show_chart("outputs/trx_charts/y_any_positive_rate.png", "Slide 5 · y_any theo 12 tháng nhãn")
show_chart("outputs/trx_charts/returning_buyer_share_monthly_comparison.png", "Slide 6 · Tỷ lệ mua quay lại theo tháng")
show_chart("outputs/trx_charts/purchase_months_per_customer.png", "Slide 7 · Số tháng có mua trên mỗi khách hàng")